Lexique des métriques :

- MCP: mesure la confiance de la classe prédite.
  - Si > 0.8 confiant, autour de 0.5 doute fort.
  - Permet de voir si le modèle est sur de lui (même quand il se trompe).

- Entropie: mesure l'incertitude de la prédiction.
  - Proche de 0 = prédiction nette, proche de 0.69 = prédiction très incertaine.
  - Permet de quantifier le doute moyen du modèle sur un scénario.

- ECE (Expected Calibration Error): écart entre confiance annoncée et précision réelle.
  - Proche de 0 = bonne calibration, plus c'est grand = plus la confiance est trompeuse.
  - Règle pratique: ECE faible (ex. < 0.05) = calibration plutot correcte; ECE élevé (ex. > 0.1) = sur/sous-confiance notable.
  - Permet de vérifier si "90% de confiance" correspond vraiment à ~90% de bonnes prédictions.

### Initialisation du notebook

Les imports nécessaires

In [16]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split


# Fonction utilitaire pour la calibration (ECE)
def compute_ece_local(confidences, predictions, targets, n_bins=10):
    confidences = np.asarray(confidences)
    predictions = np.asarray(predictions).astype(int)
    targets = np.asarray(targets).astype(int)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        left, right = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= left) & (confidences <= right)
        else:
            mask = (confidences >= left) & (confidences < right)

        if not np.any(mask):
            continue

        bin_conf = confidences[mask].mean()
        bin_acc = (predictions[mask] == targets[mask]).mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)

    return ece

Préparation des données

In [17]:
# Chargement des attributs
p_attr = pd.read_json("data/Face4Shifts/Anno/p_attr.json", lines=True)

# Construction de la variable cible proxy
p_attr["label"] = (p_attr["long_hair"] == 1) & ((p_attr["smile_with_closed_lips"] == 1) | (p_attr["smile_with_open_lips"] == 1))

X = p_attr.drop(columns=["label"])
y = p_attr["label"]

# Création de la variable de stratification (cf notebook d'Andrew)
X["label_gender"] = y.astype(str) + "_" + X["gender"].astype(str)

# Split avec Stratify et test_size=0.25
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X["label_gender"])
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=X_train_full["label_gender"])


print(f"Tailles : Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

Tailles : Train=18000, Val=6000, Test=6000


Définition du Dataset et des transformations

In [18]:
class FaceDataset(Dataset):
    def __init__(self, img_dir, data, label, transform=None):
        self.img_dir = img_dir
        self.data = data
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx]["ID"].strip()
        label = self.label.iloc[idx]

        # Gère les datasets contenant un mélange de formats d'images
        extensions = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
        img_path = None
        for ext in extensions:
            candidate = os.path.join(self.img_dir, f"{img_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            raise FileNotFoundError(
                f"Aucune image trouvée pour ID={img_id} dans {self.img_dir} "
                f"(extensions testées: .jpg, .jpeg, .png)"
            )

        # Ouvrir l'image et convertir tous les formats en RGB
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# Transformations standard pour EfficientNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Création du DataLoader de TEST
test_dataset = FaceDataset(img_dir="data/Face4Shifts/Img/Photo", data=X_test, label=y_test, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Chargement du modèle pré-entraîné

In [19]:
# Chemin vers ton modèle sauvegardé
MODEL_PATH = "face_effnet_b0_ss_reweight.pth"

# Configuration du device (GPU/MPS/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Device utilisé : {device}")

# Initialisation de l'architecture EfficientNet-b0
model = models.efficientnet_b0(weights=None) # Pas besoin des poids ImageNet puisqu'on charge les notres
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

# Chargement de tes poids
if os.path.exists(MODEL_PATH):
    state_dict = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"Modèle chargé avec succès depuis {MODEL_PATH}.")
else:
    print(f"ATTENTION : Le fichier {MODEL_PATH} est introuvable !")

Device utilisé : cpu
Modèle chargé avec succès depuis face_effnet_b0_ss_reweight.pth.


### 1. Calcul de l'incertitude du modèle de base (MCP et ECE)

Nous pouvons dire qu'un modèle doute si l'entropie est **proche de 0.69** (valeur maximale en binaire, indiquant une égale probabilité) ET/OU si le MCP est **proche de 0.5** (indiquant aucune préférence claire entre les deux classes). 

À l'inverse, un modèle confiant a une entropie proche de 0 et un MCP proche de 1.

In [20]:
model.eval()

# Listes pour stocker les métriques d'incertitude
all_targets = []
all_preds = []
all_probs = []       # Probabilité brute (de 0 à 1)
all_mcps = []        # Score de confiance (MCP)
all_entropies = []   # Incertitude (Entropie)

eps = 1e-7

with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)

        # 1. Obtenir les logits (sortie brute du modèle)
        logits = model(images).squeeze()

        # 2. Transformer les logits en probabilités avec la fonction Sigmoïde
        probs = torch.sigmoid(logits)

        # Prédictions binaires classiques (seuil à 0.5)
        preds = (probs > 0.5).float()

        # 3. Calculer le MCP (Maximum Class Probability)
        # En binaire, c'est max(P(y=1), P(y=0)) soit max(p, 1-p)
        mcps = torch.maximum(probs, 1 - probs)

        # 4. Calculer l'Entropie binaire
        entropies = - (probs * torch.log(probs + eps) + (1 - probs) * torch.log(1 - probs + eps))

        # Stockage (on passe de PyTorch/GPU à Numpy/CPU)
        all_targets.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_mcps.extend(mcps.cpu().numpy())
        all_entropies.extend(entropies.cpu().numpy())

# Conversion en tableaux numpy pour l'analyse
all_targets = np.array(all_targets)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_mcps = np.array(all_mcps)
all_entropies = np.array(all_entropies)

# Sauvegarde explicite pour les comparaisons ultérieures
base_targets = all_targets.copy()
base_preds = all_preds.copy()
base_probs = all_probs.copy()
base_mcps = all_mcps.copy()
base_entropies = all_entropies.copy()

base_ece = compute_ece_local(all_mcps, all_preds, all_targets)

# Affichage des résultats de base
print("\n--- Incertitude de base (Single Network) ---")
print(f"Confiance moyenne (MCP) sur le test net : {all_mcps.mean():.4f}")
print(f"Incertitude moyenne (Entropie) sur le test net : {all_entropies.mean():.4f}")
print(f"Calibration (ECE) sur le test net : {base_ece:.4f}")


--- Incertitude de base (Single Network) ---
Confiance moyenne (MCP) sur le test net : 0.9611
Incertitude moyenne (Entropie) sur le test net : 0.0939
Calibration (ECE) sur le test net : 0.0544


MCP élevé et entropie faible : le modèle est sûr de lui.

ECE faible : modèle bien calibré sur ces données d'origine.

Ce sont des mesures tests qui vont permettre de comparer l'évolution de la confiance avec différentes méthodes (Distribution Shift & MC Dropout).

### 2. Calcul de l'incertitude après perturbation

#### 2.1. Méthode MC Dropout

In [21]:
# 1. Fonction pour forcer l'activation du Dropout pendant l'évaluation
def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train() # Force le dropout à s'activer

model.eval()
enable_dropout(model)

# 2. Variables pour le MC Dropout
T = 10 # Nombre d'inférences par image (la taille de notre "ensemble")
all_mc_probs = []

print("Lancement du MC Dropout...")

with torch.no_grad():
    for images, labels in test_dataloader_noisy:
        images = images.to(device)

        batch_probs = []
        # On fait passer le même batch T fois dans le modèle
        for _ in range(T):
            logits = model(images).squeeze()
            probs = torch.sigmoid(logits)
            batch_probs.append(probs)

        # On empile les T prédictions (Shape: [T, batch_size])
        batch_probs = torch.stack(batch_probs)

        # Probabilite predictive moyenne
        mean_probs = batch_probs.mean(dim=0)
        all_mc_probs.extend(mean_probs.cpu().numpy())

# 3. Calcul de l'incertitude sur ces probabilités moyennées
all_mc_probs = np.array(all_mc_probs)

# Le nouveau MCP
mc_mcps = np.maximum(all_mc_probs, 1 - all_mc_probs)

# La nouvelle Entropie
eps = 1e-7
mc_entropies = - (all_mc_probs * np.log(all_mc_probs + eps) + (1 - all_mc_probs) * np.log(1 - all_mc_probs + eps))

mc_preds = (all_mc_probs > 0.5).astype(int)
mc_ece = compute_ece_local(mc_mcps, mc_preds, y_test.to_numpy())

print("\n--- Incertitude MC Dropout sur données perturbées ---")
print(f"Confiance moyenne (MCP) : {mc_mcps.mean():.4f}")
print(f"Incertitude moyenne (Entropie) : {mc_entropies.mean():.4f}")
print(f"Calibration (ECE) après dropout : {mc_ece:.4f}")

Lancement du MC Dropout...

--- Incertitude MC Dropout sur données perturbées ---
Confiance moyenne (MCP) : 0.9360
Incertitude moyenne (Entropie) : 0.1536
Calibration (ECE) après dropout : 0.0582


Le modèle reste très confiant et la calibration se dégrade un peu.

Remarques :

- Taux de Dropout

Le taux de dropout $p$ (souvent défini par défaut à 0.2 ou 0.5 dans les architectures comme EfficientNet ou ResNet) agit comme un paramètre de régularisation.

Modifier légèrement ce taux peut changer le comportement de l'incertitude. 

Le choix de $p$ a un impact théorique sur la certitude du modèle.

- Nombre de prédictions par image

L'hyperparamètre $T$ a un impact sur l'évolution de l'entropie.

#### 2.2. Changement de Domaine OOD (Cartoon)

Extraction des images Cartoon.

In [22]:
# 1. Chargement des annotations du domaine OOD (Cartoon)
c_attr = pd.read_json("data/Face4Shifts/Anno/c_attr.json", lines=True)

# On recrée la même variable cible pour les cartoons
c_attr["label"] = (c_attr["long_hair"] == 1) & ((c_attr["smile_with_closed_lips"] == 1) | (c_attr["smile_with_open_lips"] == 1))

X_cartoon = c_attr.drop(columns=["label"])
y_cartoon = c_attr["label"]

print(f"Dataset Cartoon chargé avec {len(X_cartoon)} images.")

# 2. Création du DataLoader OOD
# On utilise le 'transform' normal (sans flou)
test_dataset_ood = FaceDataset(img_dir="data/Face4Shifts/Img/Cartoon", data=X_cartoon, label=y_cartoon, transform=transform)
test_dataloader_ood = DataLoader(test_dataset_ood, batch_size=32, shuffle=False)

# 3. Évaluation de l'incertitude sur ce nouveau domaine
model.eval()
all_targets_ood = []
all_probs_ood = []
all_mcps_ood = []
all_entropies_ood = []

eps = 1e-7

with torch.no_grad():
    for images, labels in test_dataloader_ood:
        images, labels = images.to(device), labels.to(device)

        logits = model(images).squeeze()
        probs = torch.sigmoid(logits)

        mcps = torch.maximum(probs, 1 - probs)
        entropies = - (probs * torch.log(probs + eps) + (1 - probs) * torch.log(1 - probs + eps))

        all_targets_ood.extend(labels.cpu().numpy())
        all_probs_ood.extend(probs.cpu().numpy())
        all_mcps_ood.extend(mcps.cpu().numpy())
        all_entropies_ood.extend(entropies.cpu().numpy())

all_targets_ood = np.array(all_targets_ood)
all_probs_ood = np.array(all_probs_ood)
all_mcps_ood = np.array(all_mcps_ood)
all_entropies_ood = np.array(all_entropies_ood)

# Sauvegarde explicite pour les comparaisons ultérieures
ood_targets = all_targets_ood.copy()
ood_probs = all_probs_ood.copy()
ood_mcps = all_mcps_ood.copy()
ood_entropies = all_entropies_ood.copy()

ood_preds = (all_probs_ood > 0.5).astype(int)
ood_ece = compute_ece_local(all_mcps_ood, ood_preds, all_targets_ood)

print("\n--- Incertitude sur le Domaine OOD (Cartoon) ---")
print(f"Confiance moyenne (MCP) sur les cartoons : {all_mcps_ood.mean():.4f}")
print(f"Incertitude moyenne (Entropie) sur les cartoons : {all_entropies_ood.mean():.4f}")
print(f"Calibration (ECE) sur les cartoons : {ood_ece:.4f}")

Dataset Cartoon chargé avec 25000 images.


/home/anfau/projets/fair/venv/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



--- Incertitude sur le Domaine OOD (Cartoon) ---
Confiance moyenne (MCP) sur les cartoons : 0.8702
Incertitude moyenne (Entropie) sur les cartoons : 0.2892
Calibration (ECE) sur les cartoons : 0.1121


La confiance baisse (de 0.93 à 0.87) et l'entropie augmente fortement (de 0.15 à 0.29).

Le modèle détecte que les données qu'il traite sont éloignées de son domaine d’entraînement.

Il est moins confiant et mal calibré sur les images Cartoon.

### 6. Croiser Incertitude (Robustesse) et Équité (Fairness)

Face à une dégradation des données, le modèle perd-il confiance de la même manière pour les Hommes et pour les Femmes ?

Cette cellule rassemble les principaux scores obtenus dans un seul tableau : F1, MCP, Entropie, ECE et support.

Les lignes `Global` resument chaque scenario, et les lignes par genre permettent de comparer les comportements du modele quand c'est pertinent.

In [23]:
from sklearn.metrics import f1_score


def make_summary_row(scenario, group, targets, predictions, confidences, entropies):
    targets = np.asarray(targets).astype(int)
    predictions = np.asarray(predictions).astype(int)
    confidences = np.asarray(confidences)
    entropies = np.asarray(entropies)

    return {
        "Scenario": scenario,
        "Groupe": group,
        "Support": len(targets),
        "F1": round(f1_score(targets, predictions, zero_division=0), 4),
        "MCP": round(confidences.mean(), 4),
        "Entropie": round(entropies.mean(), 4),
        "ECE": round(compute_ece_local(confidences, predictions, targets), 4),
    }


summary_rows = []


# 1. Scenarios globaux
base_predictions = base_preds.astype(int)
mc_dropout_predictions = (np.array(all_mc_probs) > 0.5).astype(int)
ood_predictions = (ood_probs > 0.5).astype(int)

summary_rows.append(
    make_summary_row(
        "Test normal",
        "Global",
        base_targets,
        base_predictions,
        base_mcps,
        base_entropies,
    )
)
summary_rows.append(
    make_summary_row(
        "MC Dropout + flou",
        "Global",
        y_test.to_numpy(),
        mc_dropout_predictions,
        np.array(mc_mcps),
        np.array(mc_entropies),
    )
)
summary_rows.append(
    make_summary_row(
        "Domaine OOD Cartoon",
        "Global",
        ood_targets,
        ood_predictions,
        ood_mcps,
        ood_entropies,
    )
)


# 2. Detail par genre pour MC Dropout + flou
genders_test_labels = X_test["gender"].map({1: "Male", 2: "Female"}).fillna("Inconnu").to_numpy()
for genre in sorted(pd.unique(genders_test_labels)):
    mask = genders_test_labels == genre
    summary_rows.append(
        make_summary_row(
            "MC Dropout + flou",
            genre,
            y_test.to_numpy()[mask],
            mc_dropout_predictions[mask],
            np.array(mc_mcps)[mask],
            np.array(mc_entropies)[mask],
        )
    )


# 3. Detail par genre pour le domaine OOD Cartoon
genders_cartoon_labels = X_cartoon["gender"].map({1: "Male", 2: "Female"}).fillna("Inconnu").to_numpy()
for genre in sorted(pd.unique(genders_cartoon_labels)):
    mask = genders_cartoon_labels == genre
    summary_rows.append(
        make_summary_row(
            "Domaine OOD Cartoon",
            genre,
            ood_targets[mask],
            ood_predictions[mask],
            ood_mcps[mask],
            ood_entropies[mask],
        )
    )


df_summary = pd.DataFrame(summary_rows)
display(df_summary)

,Scenario,Groupe,Support,F1,MCP,Entropie,ECE
0,Test normal,Global,6000,0.8394,0.9611,0.0939,0.0544
1,MC Dropout + flou,Global,6000,0.7943,0.9360,0.1536,0.0582
2,Domaine OOD Cartoon,Global,25000,0.2113,0.8702,0.2892,0.1121
3,MC Dropout + flou,Female,3490,0.8128,0.9171,0.1946,0.1013
4,MC Dropout + flou,Male,2510,0.2969,0.9622,0.0965,0.0060
5,Domaine OOD Cartoon,Female,12699,0.2754,0.8588,0.3108,0.1791
6,Domaine OOD Cartoon,Male,12301,0.0360,0.8820,0.2670,0.0430


Constats : 

- Constat n°1 : Un biais de performance massif et persistant (Fairness)

F1-Score : Que ce soit sous l'effet du flou ou sur un tout nouveau domaine (Cartoon), le modèle est systématiquement beaucoup plus performant pour les femmes que pour les hommes.

Avec le flou : Les femmes (0.8128) surperforment largement les hommes (0.2969).

Sur le domaine OOD Cartoon : La performance s'effondre globalement, mais le modèle conserve un F1-score résiduel pour les femmes (0.2754), tandis qu'il est quasiment incapable de prédire correctement pour les hommes (0.0360).

Pourquoi ? Comme observé précédemment, le modèle s'appuie très probablement sur un "raccourci" (ou variable proxy) appris sur le dataset d'entraînement. Face à des données perturbées ou hors-distribution (OOD), il s'effondre sur ce raccourci : il a tendance à prédire une classe par défaut (souvent négative) dès qu'il identifie un profil masculin. Cela détruit son F1-Score pour ce groupe (incapacité à trouver les vrais positifs).

- Constat n°2 : Une incertitude discriminatoire et mal calibrée (Robustesse)

Pour les Femmes : Face au distribution shift massif (Cartoon), le modèle montre de vrais signes de doute. L'entropie est la plus élevée (0.3108) et la confiance (MCP) baisse à 0.8588. Cependant, l'erreur de calibration (ECE) explose à 0.1791. Cela signifie que bien que le modèle doute plus, la confiance qu'il affiche ne correspond pas du tout à sa précision réelle. Il est "incertain, mais très mal calibré".

Pour les Hommes : Malgré des performances catastrophiques (F1 à 0.0360 sur Cartoon et 0.2969 sur le flou), le modèle garde une confiance anormalement élevée (MCP de 0.8820 sur Cartoon et 0.9622 sur le flou) et une entropie plus faible que pour les femmes (0.2670 et 0.0965). Le modèle est en situation flagrante de sur-confiance (overconfidence) : il se trompe massivement, mais il le fait avec certitude.

### CONCLUSION

Récapitulatif de ce qui a été fait dans ce notebook :

1. Modèle de base (Single Network)

Nous avons d'abord évalué le modèle sur des données proches de son domaine d'entraînement (Photo) pour obtenir un point de référence.
Objectif : mesurer la performance et la confiance "normales" avant toute perturbation.

2. Simulation de distribution shift

Nous avons ensuite testé le modèle sur des données perturbées (flou) et sur un domaine différent (Cartoon).
Objectif : vérifier la robustesse du modèle quand la distribution des données change.

3. MC Dropout

Nous avons activé le dropout à l'inférence et répété plusieurs prédictions par image pour approximer une incertitude de type bayésienne.
Objectif : voir si cette méthode aide à mieux estimer le doute du modèle dans des situations difficiles.

4. Mesures d'incertitude et de calibration

Nous avons comparé MCP, entropie et ECE dans chaque scénario.
Pourquoi :
- MCP indique le niveau de confiance de la prédiction.
- Entropie indique le niveau de doute.
- ECE indique si la confiance annoncée correspond vraiment à la précision observée.

5. Croisement robustesse et équité (fairness)

Nous avons séparé les résultats par genre (Global, Male, Female) avec F1, MCP, Entropie et ECE.
Objectif : vérifier si la dégradation sous shift affecte les groupes de la même manière.